In [5]:
import pandas as pd
import numpy as np

# -------------------------------------------------------------
# 1. LOAD PROCESSED DATA FILES
# -------------------------------------------------------------
nhanes = pd.read_csv(r"C:\diabetes_prediction_project\data\03_processed\nhanes_final.csv")
brfss  = pd.read_csv(r"C:\diabetes_prediction_project\data\03_processed\brfss_final.csv")

# -------------------------------------------------------------
# 2. CORRECT HARMONIZED RACE MAPPING
# -------------------------------------------------------------
# BRFSS _RACE CODEBOOK (Correct)
# 1 = White non-Hispanic
# 2 = Black non-Hispanic
# 3 = Hispanic
# 4 = Other race (including Asian, AI/AN, Pacific, multiracial)
# 9 = Missing

brfss_race_map = {
    1: "Non-Hispanic White",
    2: "Non-Hispanic Black",
    3: "Hispanic",
    4: "Other/Multiracial",
    9: np.nan
}
brfss["Race_Label"] = brfss["Race_Ethnicity"].map(brfss_race_map)

# NHANES RIDRETH1 CODEBOOK
# 1 = Mexican American
# 2 = Other Hispanic
# 3 = NH White
# 4 = NH Black
# 5 = NH Asian
# 6 = Other / Multi

nhanes_race_map = {
    1: "Hispanic",
    2: "Hispanic",
    3: "Non-Hispanic White",
    4: "Non-Hispanic Black",
    5: "Other/Multiracial",
    6: "Other/Multiracial"
}
nhanes["Race_Label"] = nhanes["Race_Ethnicity"].map(nhanes_race_map)

# -------------------------------------------------------------
# 3. SMOKING STATUS (Human-readable)
# -------------------------------------------------------------
# NHANES: 1 = Ever, 2 = Never
nhanes["Smoking_Label"] = nhanes["Smoking_Status"].replace({
    1: "Ever Smoker (Current + Former)",
    2: "Never Smoker"
})

# BRFSS: 1 = Ever, 0 = Never
brfss["Smoking_Label"] = brfss["Smoking_Status"].replace({
    1: "Ever Smoker (Current + Former)",
    0: "Never Smoker"
})

# -------------------------------------------------------------
# 4. UTILITY FUNCTIONS
# -------------------------------------------------------------
def mean_sd(series):
    return f"{series.mean():.1f} ± {series.std():.1f}"

def count_pct(df, col, value):
    count = (df[col] == value).sum()
    pct = 100 * count / len(df)
    return f"{count} ({pct:.1f}%)"

def cat_counts(df, col, cats):
    out = {}
    for c in cats:
        count = (df[col] == c).sum()
        pct = 100 * count / len(df)
        out[c] = f"{count} ({pct:.1f}%)"
    return out

# -------------------------------------------------------------
# 5. BUILD TABLE 1
# -------------------------------------------------------------
rows = []

# Age
rows.append({
    "Characteristic": "Age, years (mean ± SD)",
    "NHANES": mean_sd(nhanes["Age"]),
    "BRFSS": mean_sd(brfss["Age"])
})

# BMI
rows.append({
    "Characteristic": "BMI, kg/m² (mean ± SD)",
    "NHANES": mean_sd(nhanes["BMI"]),
    "BRFSS": mean_sd(brfss["BMI"])
})

# Gender
rows.append({
    "Characteristic": "Male, n (%)",
    "NHANES": count_pct(nhanes, "Gender", 1),
    "BRFSS": count_pct(brfss, "Gender", 0)     # BRFSS: 0 = Male
})
rows.append({
    "Characteristic": "Female, n (%)",
    "NHANES": count_pct(nhanes, "Gender", 2),
    "BRFSS": count_pct(brfss, "Gender", 1)
})

# Race
race_categories = ["Hispanic", "Non-Hispanic White", "Non-Hispanic Black", "Other/Multiracial"]
nh_r = cat_counts(nhanes, "Race_Label", race_categories)
br_r = cat_counts(brfss, "Race_Label", race_categories)

for r in race_categories:
    rows.append({
        "Characteristic": f"{r}, n (%)",
        "NHANES": nh_r[r],
        "BRFSS": br_r[r]
    })

# Smoking
smoke_categories = ["Ever Smoker (Current + Former)", "Never Smoker"]
nh_s = cat_counts(nhanes, "Smoking_Label", smoke_categories)
br_s = cat_counts(brfss, "Smoking_Label", smoke_categories)

for s in smoke_categories:
    rows.append({
        "Characteristic": f"{s}, n (%)",
        "NHANES": nh_s[s],
        "BRFSS": br_s[s]
    })

# Physical Activity
rows.append({
    "Characteristic": "Physically Active, n (%)",
    "NHANES": count_pct(nhanes, "Physical_Activity", 1),
    "BRFSS": count_pct(brfss, "Physical_Activity", 1)
})
rows.append({
    "Characteristic": "Not Active, n (%)",
    "NHANES": count_pct(nhanes, "Physical_Activity", 0),
    "BRFSS": count_pct(brfss, "Physical_Activity", 0)
})

# Heart Attack
rows.append({
    "Characteristic": "Heart Attack (Yes), n (%)",
    "NHANES": count_pct(nhanes, "History_Heart_Attack", 1),
    "BRFSS": count_pct(brfss, "History_Heart_Attack", 1)
})

# Stroke
rows.append({
    "Characteristic": "Stroke (Yes), n (%)",
    "NHANES": count_pct(nhanes, "History_Stroke", 1),
    "BRFSS": count_pct(brfss, "History_Stroke", 1)
})

# Diabetes
rows.append({
    "Characteristic": "Diabetes (Yes), n (%)",
    "NHANES": count_pct(nhanes, "Diabetes_Outcome", 1),
    "BRFSS": count_pct(brfss, "Diabetes_Outcome", 1)
})

# -------------------------------------------------------------
# 6. EXPORT FINAL TABLE
# -------------------------------------------------------------
output_path = "./outputs/TABLE_1_baseline_characteristics.csv"

import os
os.makedirs("outputs", exist_ok=True)

table1 = pd.DataFrame(rows)
table1.to_csv(output_path, index=False)

print("Saved to:", output_path)
table1



Saved to: ./outputs/TABLE_1_baseline_characteristics.csv


,Characteristic,NHANES,BRFSS
0,"Age, years (mean ± SD)",49.0 ± 18.6,55.1 ± 17.7
1,"BMI, kg/m² (mean ± SD)",29.7 ± 7.4,28.5 ± 6.5
2,"Male, n (%)",7605 (48.5%),596991 (46.4%)
3,"Female, n (%)",8080 (51.5%),688792 (53.6%)
4,"Hispanic, n (%)",3984 (25.4%),14078 (1.1%)
5,"Non-Hispanic White, n (%)",5284 (33.7%),618605 (48.1%)
6,"Non-Hispanic Black, n (%)",3820 (24.4%),62902 (4.9%)
7,"Other/Multiracial, n (%)",1898 (12.1%),21546 (1.7%)
8,"Ever Smoker (Current + Former), n (%)",6311 (40.2%),487117 (37.9%)
9,"Never Smoker, n (%)",9358 (59.7%),717134 (55.8%)


In [2]:
# Get exact BRFSS Gender percentages
gender_dist = brfss_df['Gender'].map({0.0: 'Male', 1.0: 'Female'}).value_counts(normalize=True).mul(100)
print("BRFSS Gender Distribution:")
print(gender_dist.round(1).astype(str) + '%')

BRFSS Gender Distribution:
Gender
Female    53.6%
Male      46.4%
Name: proportion, dtype: object


In [1]:
import pandas as pd
import numpy as np
import math
from sklearn.metrics import roc_auc_score

# -------------------------------------------------------------------
# 1. Load data
# -------------------------------------------------------------------
brfss = pd.read_csv(r"C:\diabetes_prediction_project\data\03_processed\brfss_final.csv")
preds = pd.read_csv(r"C:\diabetes_prediction_project\notebooks\results\external_validation_predictions.csv")

# Keep only rows with non-missing outcome (these were used for eval)
brfss_sub = brfss[brfss["Diabetes_Outcome"].notna()].reset_index(drop=True)

# Sanity check: lengths must match
assert len(brfss_sub) == len(preds), "Row mismatch between BRFSS and predictions!"

# Sanity check: y_true must equal Diabetes_Outcome
assert (brfss_sub["Diabetes_Outcome"].values == preds["y_true"].values).all(), \
    "y_true not aligned with Diabetes_Outcome"

# Attach predictions
brfss_sub["y_true"] = preds["y_true"].values
brfss_sub["y_pred_proba"] = preds["y_pred_proba"].values

# -------------------------------------------------------------------
# 2. Labels: Gender and Race (with correct mapping)
# -------------------------------------------------------------------
# BRFSS Gender in your pipeline: 0 = Male, 1 = Female
brfss_sub["Gender_Label"] = brfss_sub["Gender"].map({0.0: "Male", 1.0: "Female"})

# Race_Ethnicity codes (as you confirmed)
# 1 = Non-Hispanic White
# 2 = Non-Hispanic Black
# 3 = Hispanic
# 4 = Other / Multiracial
# 5 = Non-Hispanic Asian
#
# For fairness table (to match your manuscript), we collapse 4 + 5:
def race_fair(code):
    if code == 1.0:
        return "White, Non-Hispanic"
    if code == 2.0:
        return "Black, Non-Hispanic"
    if code == 3.0:
        return "Hispanic"
    if code in (4.0, 5.0):
        return "Other/Multiracial"
    return np.nan

brfss_sub["Race_Fair"] = brfss_sub["Race_Ethnicity"].map(race_fair)

# -------------------------------------------------------------------
# 3. Helper functions: AUC, CI, and p-values vs reference
#    (Hanley–McNeil variance approximation; good and fast)
# -------------------------------------------------------------------
def auc_ci_stats(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    auc = roc_auc_score(y_true, y_score)
    pos = (y_true == 1).sum()
    neg = (y_true == 0).sum()

    if pos == 0 or neg == 0:
        return auc, (np.nan, np.nan), np.nan

    Q1 = auc / (2 - auc)
    Q2 = 2 * auc**2 / (1 + auc)
    var_auc = (auc * (1 - auc) +
               (pos - 1) * (Q1 - auc**2) +
               (neg - 1) * (Q2 - auc**2)) / (pos * neg)
    var_auc = max(var_auc, 0.0)
    se = math.sqrt(var_auc)
    z_95 = 1.96
    lower = max(0.0, auc - z_95 * se)
    upper = min(1.0, auc + z_95 * se)
    return auc, (lower, upper), se

def normal_cdf(x):
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def p_value_vs_ref(auc_ref, se_ref, auc_grp, se_grp):
    # z-test on difference in AUCs
    var = se_ref**2 + se_grp**2
    if var <= 0:
        return np.nan
    z = (auc_grp - auc_ref) / math.sqrt(var)
    p = 2 * (1 - normal_cdf(abs(z)))
    return p

def format_ci(ci):
    if any(np.isnan(ci)):
        return "NA"
    return f"[{ci[0]:.3f}, {ci[1]:.3f}]"

def format_p(p):
    if pd.isna(p):
        return "-"
    if p < 0.001:
        return "<0.001"
    return f"{p:.3f}"

# -------------------------------------------------------------------
# 4. Compute subgroup AUCs
# -------------------------------------------------------------------
rows = []

# ---- GENDER ----
gender_groups = ["Female", "Male"]
# Reference: Male
ref_gender = "Male"
ref_df = brfss_sub[brfss_sub["Gender_Label"] == ref_gender]
auc_ref, ci_ref, se_ref = auc_ci_stats(ref_df["y_true"], ref_df["y_pred_proba"])

for g in gender_groups:
    sub = brfss_sub[brfss_sub["Gender_Label"] == g]
    auc, ci, se = auc_ci_stats(sub["y_true"], sub["y_pred_proba"])
    if g == ref_gender:
        p = np.nan
    else:
        p = p_value_vs_ref(auc_ref, se_ref, auc, se)

    rows.append({
        "Subgroup": "Gender",
        "Category": g,
        "n": f"{len(sub):,}",
        "AUC": round(auc, 3),
        "95% CI": format_ci(ci),
        "p-value*": format_p(p)
    })

# ---- RACE / ETHNICITY ----
race_groups = ["White, Non-Hispanic", "Black, Non-Hispanic",
               "Hispanic", "Other/Multiracial"]
ref_race = "White, Non-Hispanic"
ref_df = brfss_sub[brfss_sub["Race_Fair"] == ref_race]
auc_ref, ci_ref, se_ref = auc_ci_stats(ref_df["y_true"], ref_df["y_pred_proba"])

for r in race_groups:
    sub = brfss_sub[brfss_sub["Race_Fair"] == r]
    auc, ci, se = auc_ci_stats(sub["y_true"], sub["y_pred_proba"])
    if r == ref_race:
        p = np.nan
    else:
        p = p_value_vs_ref(auc_ref, se_ref, auc, se)

    rows.append({
        "Subgroup": "Race/Ethnicity",
        "Category": r,
        "n": f"{len(sub):,}",
        "AUC": round(auc, 3),
        "95% CI": format_ci(ci),
        "p-value*": format_p(p)
    })

# -------------------------------------------------------------------
# 5. Build and save TABLE 4
# -------------------------------------------------------------------
fairness_table = pd.DataFrame(rows)

fairness_table.to_csv("TABLE_4_fairness_analysis_corrected.csv", index=False)
print("Saved TABLE_4_fairness_analysis_corrected.csv")
fairness_table


Saved TABLE_4_fairness_analysis_corrected.csv


,Subgroup,Category,n,AUC,95% CI,p-value*
0,Gender,Female,"687,435",0.712,"[0.710, 0.714]",<0.001
1,Gender,Male,"595,462",0.723,"[0.721, 0.725]",-
2,Race/Ethnicity,"White, Non-Hispanic","617,782",0.742,"[0.740, 0.744]",-
3,Race/Ethnicity,"Black, Non-Hispanic","62,770",0.728,"[0.723, 0.734]",<0.001
4,Race/Ethnicity,Hispanic,"14,042",0.726,"[0.714, 0.737]",0.006
5,Race/Ethnicity,Other/Multiracial,"25,453",0.800,"[0.790, 0.810]",<0.001
